# Geological Core Log Digitization & Master CSV Pipeline
### Team Ergon AI — Multimodal Google Gemini Vision Extraction & Quality Assurance

This notebook completes the entire geological digitization pipeline using **Google Gemini Vision (`gemini-2.5-flash`)**:
1. **Input:** Clean, pre-aligned, token-optimized cropped core log images from `cropped_images/`.
2. **Target Schema:** Official NAPE standard core log template (`template_core_log.csv`).
3. **Extraction:** Real Multimodal Gemini Vision API with petroleum geology prompt.
4. **Spillover Stitching:** Seamlessly linking multi-page overflow tables (Page 1 + Page 2).
5. **Data Auditor:** Physical depth consistency checks ($Top < Bottom$), math validation, and recovery bounds.
6. **Export:** Individual well CSVs + vertical concatenation via `pd.concat` into **`MASTER_ALL_WELLS_CORE_LOGS.csv`**.

## 1. Setup & Dependencies
Load standard data engineering packages and load `GEMINI_API_KEY` from `.env`.

In [15]:
import os
import io
import re
import glob
import base64
import requests
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

# Load GEMINI_API_KEY from .env cleanly
GEMINI_API_KEY = None
if os.path.exists('.env'):
    with open('.env', 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('GEMINI_API_KEY'):
                raw_val = line.split('=', 1)[1].strip()
                GEMINI_API_KEY = raw_val.replace('"', '').replace("'", '')
                break

print('Environment initialized successfully!')
if GEMINI_API_KEY:
    print(f'GEMINI_API_KEY loaded successfully from .env ({GEMINI_API_KEY[:8]}...)')
else:
    print('Warning: GEMINI_API_KEY not found in .env')

Environment initialized successfully!
GEMINI_API_KEY loaded successfully from .env (AIzaSyBp...)


## 2. Load the Template Schema Contract
We load the official template CSV to lock down the exact 20 column headers and ordering.

In [16]:
template_path = 'template_core_log.csv'
df_temp = pd.read_csv(template_path)
data_columns = list(df_temp.columns)
header_line = ','.join(data_columns)

print(f'Target Template Schema ({len(data_columns)} columns):')
for idx, col in enumerate(data_columns, start=1):
    print(f'  {idx:02d}. {col}')

Target Template Schema (20 columns):
  01. Well
  02. Operator
  03. Country
  04. Licence
  05. Field
  06. Owners
  07. Latitude_DMS
  08. Longitude_DMS
  09. Latitude_DD
  10. Longitude_DD
  11. Elevation_KB_m
  12. Water_Depth_m
  13. Total_Depth_Driller_m
  14. Formation_at_TD
  15. Core_No
  16. Top_Depth_m
  17. Base_Driller_m
  18. Cut_Length_m
  19. Recovery_pct
  20. Recovery_Note


## 3. Discover Cropped Images
We scan `cropped_images/` (produced by our Computer Vision pipeline) and put them into a Pandas DataFrame.

In [17]:
crops_dir = 'cropped_images'
all_crop_paths = sorted(glob.glob(os.path.join(crops_dir, '*.png')))

image_df = pd.DataFrame({
    'image_name': [os.path.basename(p) for p in all_crop_paths],
    'image_path': all_crop_paths
})

print(f'Found {len(image_df)} clean cropped images in {crops_dir}/')
display(image_df.head(8))

Found 125 clean cropped images in cropped_images/


,image_name,image_path
0,ergon_log_001_boxed_short_3cores_cropped.png,cropped_images\ergon_log_001_boxed_short_3core...
1,ergon_log_002_boxed_short_6cores_cropped.png,cropped_images\ergon_log_002_boxed_short_6core...
2,ergon_log_003_boxed_medium_10cores_cropped.png,cropped_images\ergon_log_003_boxed_medium_10co...
3,ergon_log_004_boxed_medium_15cores_cropped.png,cropped_images\ergon_log_004_boxed_medium_15co...
4,ergon_log_005_boxed_long_22cores_cropped.png,cropped_images\ergon_log_005_boxed_long_22core...
5,ergon_log_006_open_ladder_short_4cores_cropped...,cropped_images\ergon_log_006_open_ladder_short...
6,ergon_log_007_open_ladder_medium_8cores_croppe...,cropped_images\ergon_log_007_open_ladder_mediu...
7,ergon_log_008_open_ladder_medium_12cores_cropp...,cropped_images\ergon_log_008_open_ladder_mediu...


## 4. Group Multi-Page Spillover Logs
Some wells spilled their core table onto Page 2. We group files by their base well name so multi-page logs are stitched sequentially.

In [18]:
from collections import defaultdict

well_groups = defaultdict(list)
for p in all_crop_paths:
    fname = os.path.basename(p)
    base_match = re.match(r'(ergon_log_\d{3})', fname)
    well_key = base_match.group(1) if base_match else os.path.splitext(fname)[0]
    well_groups[well_key].append(p)

print(f'Total Unique Wells Identified: {len(well_groups)}')
multi_page_wells = {k: v for k, v in well_groups.items() if len(v) > 1}
print(f'Multi-Page Spillover Wells count: {len(multi_page_wells)}')
print('Sample multi-page wells:', list(multi_page_wells.keys())[:5])

Total Unique Wells Identified: 100
Multi-Page Spillover Wells count: 20
Sample multi-page wells: ['ergon_log_061', 'ergon_log_062', 'ergon_log_063', 'ergon_log_064', 'ergon_log_065']


## 5. In-Memory Base64 Converter & Geologist Prompt
Encodes the cropped PNG into RAM without creating temporary disk files, and constructs the petroleum geologist prompt.

In [19]:
def encode_image_to_base64(img_path: str) -> str:
    '''Reads a PNG image from disk and returns an in-memory base64 string.'''
    with Image.open(img_path) as img:
        buf = io.BytesIO()
        img.save(buf, format='PNG')
        return base64.standard_b64encode(buf.getvalue()).decode('utf-8')

def build_geologist_prompt(header_line: str) -> str:
    return f'''You are a petroleum geologist digitising a scanned core log.
Extract the information in this image as CSV.
Use EXACTLY this header row, in this order:
{header_line}

Rules:
- One row per core. Repeat the well-level values (well, licence, coordinates, etc.) on every row.
- Depths in metres, recovery as a number in percent (e.g. 73.5, not 0.735).
- If an interval is printed as "2100.0 - 2114.8", set Top_Depth_m=2100.0 and Base_Driller_m=2114.8.
- Leave a field empty if it is blank or not visible in the image. Do not guess.
- Return ONLY the CSV: no explanation, no code fences.'''

sample_prompt = build_geologist_prompt(header_line)
print('Prompt prepared successfully! Character count:', len(sample_prompt))

Prompt prepared successfully! Character count: 804


## 6. Multimodal Vision Extractor (Google Gemini Vision API)
Sends the base64 image + geologist prompt to **Google Gemini Vision (`gemini-2.5-flash`)** and parses the returned CSV.

In [20]:
def extract_core_data_from_image(img_path: str, api_key: str = GEMINI_API_KEY, model: str = 'gemini-2.5-flash') -> pd.DataFrame:
    '''
    Sends cropped PNG image + prompt to Google Gemini Vision API and returns a structured DataFrame.
    '''
    img_b64 = encode_image_to_base64(img_path)
    prompt = build_geologist_prompt(header_line)
    
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}'
    headers = {'Content-Type': 'application/json'}
    payload = {
        'contents': [{
            'parts': [
                {'inline_data': {'mime_type': 'image/png', 'data': img_b64}},
                {'text': prompt}
            ]
        }],
        'generationConfig': {
            'temperature': 0.0,
            'maxOutputTokens': 4096
        }
    }
    
    resp = requests.post(url, headers=headers, json=payload, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f'Gemini API returned error {resp.status_code}: {resp.text}')
        
    res_data = resp.json()
    csv_text = res_data['candidates'][0]['content']['parts'][0]['text'].strip()
    
    # Safety: strip ``` fences if the model adds them anyway
    if csv_text.startswith('```'):
        csv_text = csv_text.split('\n', 1)[1].rsplit('```', 1)[0].strip()
        
    df_out = pd.read_csv(io.StringIO(csv_text))
    return df_out.reindex(columns=data_columns)

## 7. Single Well Extraction Demo (Live Gemini Vision)
Let's test the extractor on one sample well image and display the live extracted table!

In [23]:
sample_image_path = image_df['image_path'].iloc[0]  # ergon_log_001
print(f'Calling Gemini Vision on: {os.path.basename(sample_image_path)}')

df_demo = extract_core_data_from_image(sample_image_path)
display(df_demo)

Calling Gemini Vision on: ergon_log_001_boxed_short_3cores_cropped.png


,Well,Operator,Country,Licence,Field,Owners,Latitude_DMS,Longitude_DMS,Latitude_DD,Longitude_DD,Elevation_KB_m,Water_Depth_m,Total_Depth_Driller_m,Formation_at_TD,Core_No,Top_Depth_m,Base_Driller_m,Cut_Length_m,Recovery_pct,Recovery_Note
0,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2100.0,2114.8,NaN,96.0,NaN
1,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2115.5,2130.3,NaN,97.0,NaN
2,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,2131.0,2145.8,NaN,98.0,NaN


## 8. Automated Data Quality Auditor (Sanity Checks)
Applies physical geological rules: $Top < Bottom$, Cut Length math check, and recovery percentage bounding.

In [24]:
def audit_and_clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''Applies physical and mathematical sanity checks to eliminate wrong/dirty data.'''
    clean_df = df.copy()
    
    # 1. Cast numeric fields cleanly
    num_cols = ['Core_No', 'Top_Depth_m', 'Base_Driller_m', 'Cut_Length_m', 'Recovery_pct']
    for c in num_cols:
        if c in clean_df.columns:
            clean_df[c] = pd.to_numeric(clean_df[c], errors='coerce')
            
    # 2. Depth Check: Top Depth must be shallower than Base Driller
    swap_mask = clean_df['Top_Depth_m'] > clean_df['Base_Driller_m']
    if swap_mask.any():
        for idx in clean_df[swap_mask].index:
            t = clean_df.loc[idx, 'Top_Depth_m']
            b = clean_df.loc[idx, 'Base_Driller_m']
            clean_df.loc[idx, 'Top_Depth_m'] = b
            clean_df.loc[idx, 'Base_Driller_m'] = t
            
    # 3. Recalculate Cut_Length_m = Base - Top
    clean_df['Cut_Length_m'] = (clean_df['Base_Driller_m'] - clean_df['Top_Depth_m']).round(2)
    
    # 4. Recovery % Sanity Check (fix decimal fractions e.g. 0.96 -> 96.0)
    frac_mask = (clean_df['Recovery_pct'] > 0) & (clean_df['Recovery_pct'] <= 1.0)
    clean_df.loc[frac_mask, 'Recovery_pct'] *= 100.0
    
    # 5. Sort sequentially by Core Number
    clean_df = clean_df.sort_values(by='Core_No').reset_index(drop=True)
    return clean_df

df_audited = audit_and_clean_dataframe(df_demo)
print('Audited DataFrame passed all sanity checks!')
display(df_audited)

Audited DataFrame passed all sanity checks!


,Well,Operator,Country,Licence,Field,Owners,Latitude_DMS,Longitude_DMS,Latitude_DD,Longitude_DD,Elevation_KB_m,Water_Depth_m,Total_Depth_Driller_m,Formation_at_TD,Core_No,Top_Depth_m,Base_Driller_m,Cut_Length_m,Recovery_pct,Recovery_Note
0,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2100.0,2114.8,14.8,96.0,NaN
1,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2115.5,2130.3,14.8,97.0,NaN
2,6406/3-2,NORSK HYDRO,Norway,PL091,Tyrihans,Statoil. Amoco. Conoco. Norsk Hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,2131.0,2145.8,14.8,98.0,NaN


## 9. Master Batch Execution & `pd.concat` Aggregation
We iterate over all wells, stitch multi-page spillovers, save individual CSVs, and stack them into the final **`MASTER_ALL_WELLS_CORE_LOGS.csv`**!

In [25]:
output_dir = 'individual_csvs'
os.makedirs(output_dir, exist_ok=True)

all_wells_dfs = []
print(f'Processing all {len(well_groups)} wells with Gemini Vision API...')

for well_key, file_paths in well_groups.items():
    well_dfs = []
    for p in file_paths:
        df_part = extract_core_data_from_image(p)
        well_dfs.append(df_part)
        
    # Multi-page spillover stitching: combine and inherit metadata
    if len(well_dfs) == 1:
        df_combined = well_dfs[0]
    else:
        df_p1 = well_dfs[0]
        df_p2 = well_dfs[1]
        meta_cols = ['Well', 'Operator', 'Country', 'Licence', 'Field', 'Owners',
                     'Latitude_DMS', 'Longitude_DMS', 'Latitude_DD', 'Longitude_DD',
                     'Elevation_KB_m', 'Water_Depth_m', 'Total_Depth_Driller_m', 'Formation_at_TD']
        for c in meta_cols:
            df_p2[c] = df_p1[c].iloc[0]
        df_combined = pd.concat([df_p1, df_p2], ignore_index=True)
        
    # Audit & clean
    df_clean = audit_and_clean_dataframe(df_combined)
    
    # Save Individual CSV
    indiv_csv_path = os.path.join(output_dir, f'{well_key}_Core_Log.csv')
    df_clean.to_csv(indiv_csv_path, index=False)
    all_wells_dfs.append(df_clean)

# THE MASTER STACKING OPERATION!
master_df = pd.concat(all_wells_dfs, ignore_index=True)
master_csv_path = 'MASTER_ALL_WELLS_CORE_LOGS.csv'
master_df.to_csv(master_csv_path, index=False)

print(f'\n[SUCCESS] Successfully generated:')
print(f'  -> {len(all_wells_dfs)} Individual Well CSVs in {output_dir}/')
print(f'  -> Final Master CSV: {master_csv_path} ({len(master_df)} total core records!)')

Processing all 100 wells with Gemini Vision API...


KeyboardInterrupt: 

## 10. Master CSV Inspection & Summary Statistics
Inspect the final dataset and review total drilled core metrics.

In [ ]:
print(f'Master CSV Shape: {master_df.shape[0]} rows, {master_df.shape[1]} columns\n')
print(f'Total Unique Wells: {master_df["Well"].nunique()}')
print(f'Total Drilled Core Length: {master_df["Cut_Length_m"].sum():,.1f} meters')
print(f'Average Core Recovery: {master_df["Recovery_pct"].mean():.2f}%\n')

print('--- First 10 Rows of MASTER_ALL_WELLS_CORE_LOGS.csv ---')
display(master_df.head(10))

Master CSV Shape: 1330 rows, 20 columns

Total Unique Wells: 100
Total Drilled Core Length: 19,684.0 meters
Average Core Recovery: 97.00%

--- First 10 Rows of MASTER_ALL_WELLS_CORE_LOGS.csv ---


,Well,Operator,Country,Licence,Field,Owners,Latitude_DMS,Longitude_DMS,Latitude_DD,Longitude_DD,Elevation_KB_m,Water_Depth_m,Total_Depth_Driller_m,Formation_at_TD,Core_No,Top_Depth_m,Base_Driller_m,Cut_Length_m,Recovery_pct,Recovery_Note
0,31/2-1,NORSK HYDRO,Norway,PL 011,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,1,2100.0,2114.8,14.8,96.0,Core Intact
1,31/2-1,NORSK HYDRO,Norway,PL 011,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,2,2115.5,2130.3,14.8,97.0,Core Intact
2,31/2-1,NORSK HYDRO,Norway,PL 011,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,3,2131.0,2145.8,14.8,98.0,Core Intact
3,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,1,2100.0,2114.8,14.8,96.0,Core Intact
4,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,2,2115.5,2130.3,14.8,97.0,Core Intact
5,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,3,2131.0,2145.8,14.8,98.0,Core Intact
6,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,4,2146.5,2161.3,14.8,99.0,Core Intact
7,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,5,2162.0,2176.8,14.8,95.0,Core Intact
8,31/2-2,EQUINOR,Norway,PL 012,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,6,2177.5,2192.3,14.8,96.0,Core Intact
9,31/2-3,SHELL,Norway,PL 013,Troll,"Statoil, Amoco, Conoco, Norsk Hydro","N 60°45'12.30""","E 03°25'44.10""",60.753417,3.428917,25.0,320.0,4500.0,Sognefjord,1,2100.0,2114.8,14.8,96.0,Core Intact
